In [ ]:
import numpy as np
import cv2
import torchvision.io as io
import matplotlib.pyplot as plt

In [ ]:
img_path = "../notebooks/cat.jpg"
# load image as torch tensor
img_tensor = io.read_image(img_path)
img_np = img_tensor.permute(1, 2, 0).numpy()
plt.imshow(img_np)
plt.axis('off')
plt.show()

## Motion Blur

In [ ]:
def _motion_blur(img_uint8: np.ndarray, severity: int) -> np.ndarray:
    """Apply directional motion blur."""
    if cv2 is None:
        raise ImportError("opencv-python is required for motion_blur")

    kernel_sizes = [5, 9, 13, 17, 21]
    k = kernel_sizes[severity - 1]

    kernel = np.zeros((k, k), dtype=np.float32)
    kernel[k // 2, :] = 1.0
    kernel /= k

    return cv2.filter2D(img_uint8, -1, kernel),kernel

In [ ]:
motion_blurred,kernel = _motion_blur(img_np, severity=5)
for row in kernel: print(row)
plt.imshow(motion_blurred)
plt.axis('off')
plt.show()

## Gaussian

In [ ]:
def _gaussian_blur(img_uint8: np.ndarray, severity: int) -> np.ndarray:
    """Apply Gaussian blur."""
    if cv2 is None:
        raise ImportError("opencv-python is required for gaussian_blur")

    sigmas = [1, 2, 3, 4, 5]
    sigma = sigmas[severity - 1]
    ksize = int(2 * round(2 * sigma) + 1)

    return cv2.GaussianBlur(img_uint8, (ksize, ksize), sigma)

In [ ]:
gaussian_blurred = _gaussian_blur(img_np, severity=5)
plt.imshow(gaussian_blurred)
plt.axis('off')
plt.show()

## zoom

In [ ]:
def _zoom_blur(img_uint8: np.ndarray, severity: int) -> np.ndarray:
    """Simulate zoom blur by averaging zoomed-in copies."""
    zoom_factors = [
        [1.02, 1.04],
        [1.04, 1.08],
        [1.06, 1.12, 1.18],
        [1.08, 1.16, 1.24],
        [1.10, 1.20, 1.30, 1.40],
    ]
    factors = zoom_factors[severity - 1]

    h, w, c = img_uint8.shape
    img_f = img_uint8.astype(np.float32)
    accum = img_f.copy()

    for factor in factors:
        # Centre-crop a zoomed version back to original size
        zh, zw = int(h * factor), int(w * factor)
        if cv2 is not None:
            zoomed = cv2.resize(img_uint8, (zw, zh), interpolation=cv2.INTER_LINEAR)
        elif scipy_zoom is not None:
            zoomed = scipy_zoom(img_uint8, (factor, factor, 1), order=1).astype(np.uint8)
        else:
            # Fallback: no zoom, just accumulate original
            accum += img_f
            continue

        # Centre crop
        y0 = (zh - h) // 2
        x0 = (zw - w) // 2
        cropped = zoomed[y0 : y0 + h, x0 : x0 + w]

        # Handle potential off-by-one from rounding
        if cropped.shape[0] != h or cropped.shape[1] != w:
            cropped = cv2.resize(cropped, (w, h)) if cv2 is not None else cropped[:h, :w]

        accum += cropped.astype(np.float32)

    result = accum / (1 + len(factors))
    return np.clip(result, 0, 255).astype(np.uint8)

In [ ]:
zoom_blurred = _zoom_blur(img_np, severity=5)
plt.imshow(zoom_blurred)
plt.axis('off')
plt.show()

## Fog

In [ ]:
def _fog(img_uint8: np.ndarray, severity: int) -> np.ndarray:
    """Overlay a foggy haze."""
    strengths = [0.3, 0.45, 0.6, 0.75, 0.9]
    strength = strengths[severity - 1]

    h, w, c = img_uint8.shape
    img_f = img_uint8.astype(np.float32) / 255.0

    # Create a smooth fog pattern using low-frequency noise
    rng = np.random.RandomState(42)  # deterministic fog
    fog_map = rng.rand(h // 8 + 1, w // 8 + 1).astype(np.float32)

    if cv2 is not None:
        fog_map = cv2.resize(fog_map, (w, h), interpolation=cv2.INTER_LINEAR)
    else:
        # Simple repeat upscale as fallback
        fog_map = np.repeat(np.repeat(fog_map, 8, axis=0), 8, axis=1)[:h, :w]

    fog_map = fog_map[..., np.newaxis]  # (H, W, 1) 

    # Blend with white fog
    result = img_f * (1 - strength * fog_map) + strength * fog_map
    return np.clip(result * 255, 0, 255).astype(np.uint8)


In [ ]:
fog_blurred = _fog(img_np, severity=5)
plt.imshow(fog_blurred)
plt.axis('off')
plt.show()

## Glass

In [ ]:
def _glass_blur(img_uint8: np.ndarray, severity: int) -> np.ndarray:
    """Simulate glass-like distortion via local pixel shuffling + blur."""
    if cv2 is None:
        raise ImportError("opencv-python is required for glass_blur")

    sigmas = [0.5, 0.7, 0.9, 1.1, 1.5]
    deltas = [1, 1, 2, 2, 3]
    iters_list = [1, 2, 2, 2, 2]

    sigma = sigmas[severity - 1]
    delta = deltas[severity - 1]
    n_iters = iters_list[severity - 1]

    h, w, c = img_uint8.shape
    result = cv2.GaussianBlur(img_uint8, (0, 0), sigma).astype(np.float32)

    rng = np.random.RandomState(42)

    for _ in range(n_iters):
        for i in range(h - delta, delta, -1):
            for j in range(w - delta, delta, -1):
                di = rng.randint(-delta, delta + 1)
                dj = rng.randint(-delta, delta + 1)
                ni = min(max(i + di, 0), h - 1)
                nj = min(max(j + dj, 0), w - 1)
                result[i, j], result[ni, nj] = (
                    result[ni, nj].copy(),
                    result[i, j].copy(),
                )

    result = cv2.GaussianBlur(result.astype(np.uint8), (0, 0), sigma)
    return result

In [ ]:
glass_blurred = _glass_blur(img_np, severity=5)
plt.imshow(glass_blurred)
plt.axis('off')
plt.show()

In [ ]:
# make a 3x2 plot of the 5 blurs with severity 5 and the original image
fig, axs = plt.subplots(2, 3, figsize=(15, 10))
axs[0, 0].imshow(img_np)
axs[0, 0].set_title("Original")
axs[0, 1].imshow(motion_blurred)
axs[0, 1].set_title("Motion Blur")
axs[0, 2].imshow(gaussian_blurred)
axs[0, 2].set_title("Gaussian Blur")
axs[1, 0].imshow(zoom_blurred)
axs[1, 0].set_title("Zoom Blur")
axs[1, 1].imshow(fog_blurred)
axs[1, 1].set_title("Fog Blur")
axs[1, 2].imshow(glass_blurred)
axs[1, 2].set_title("Glass Blur")
for ax in axs.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()